In [ ]:
%pip install keras-tcn openpyxl --quiet

In [ ]:
import os,time,warnings,gc,random
from collections import defaultdict
import numpy as np
import pandas as pd
import glob
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d
from scipy.signal import find_peaks
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers,Model
from tcn import TCN
warnings.filterwarnings('ignore')
SEED=42
os.environ['PYTHONHASHSEED']=str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)
def _set_seed(s=SEED):
    random.seed(s); np.random.seed(s)
    tf.random.set_seed(s); keras.utils.set_random_seed(s)
print(f'TF {tf.__version__}')

In [ ]:
EXERCISE=1
_CFG={1: dict(n_samples=213,label_col='clinical TS Ex#1',dropout=0.05),2: dict(n_samples=183,label_col='clinical TS Ex#2',dropout=0.05),3: dict(n_samples=189,label_col='clinical TS Ex#3',dropout=0.05),4: dict(n_samples=210,label_col='clinical TS Ex#4',dropout=0.05),5: dict(n_samples=207,label_col='clinical TS Ex#5',dropout=0.005),}
cfg=_CFG[EXERCISE]
N_SAMPLES=cfg['n_samples']
LABEL_COL=cfg['label_col']
DROPOUT=cfg['dropout']
EX_TAG=f'EX{EXERCISE}'
ROOT='/kaggle/input/datasets/aadharshramachandran/kimore/Kimore/Kimore'
WORK_DIR='/kaggle/working'
NPY_X=os.path.join(WORK_DIR,f'X_{EX_TAG}.npy')
NPY_Y=os.path.join(WORK_DIR,f'y_{EX_TAG}.npy')
CKPT=os.path.join(WORK_DIR,f'LightPRA_XAT_{EX_TAG}.weights.h5')
print(f'Exercise : {EX_TAG}  |  Expected N: {N_SAMPLES}  |  Dropout: {DROPOUT}')

In [ ]:
PROBLEMATIC={1: {'P_ID8','P_ID17','P_ID52','P_ID21','P_ID55','P_ID73','P_ID57'},2: {'P_ID4','P_ID8','P_ID11','P_ID17','P_ID20','P_ID21','P_ID22','P_ID25','P_ID27','P_ID30','P_ID33','P_ID36','P_ID42','P_ID52','P_ID55','P_ID73','P_ID57'},3: {'P_ID4','P_ID8','P_ID11','P_ID17','P_ID20','P_ID21','P_ID22','P_ID27','P_ID30','P_ID33','P_ID36','P_ID52','P_ID55','P_ID57','P_ID73'},4: {'P_ID8','P_ID17','P_ID21','P_ID52','P_ID55','P_ID57','P_ID73','P_ID27'},5: {'P_ID8','P_ID17','P_ID21','P_ID52','P_ID55','P_ID57','P_ID73','P_ID27','P_ID4'}}
COL_ORDER=(list(range(0,8))+list(range(16,20))+list(range(32,36)) +list(range(20,32))+list(range(84,88)) +list(range(36,48))+list(range(92,96)) +list(range(48,64))+list(range(64,80)))
seen=set()
COL_ORDER_CLEAN=[]
for c in COL_ORDER:
    if c < 100 and c not in seen:
        COL_ORDER_CLEAN.append(c); seen.add(c)
remaining=[c for c in range(100) if c not in seen]
COL_ORDER_CLEAN=(COL_ORDER_CLEAN+remaining)[:80]
assert len(COL_ORDER_CLEAN)==80
EXERCISE_CHANNELS={1:48,2:48,3:8,4:52,5:52}
print(f'COL_ORDER_CLEAN length: {len(COL_ORDER_CLEAN)}')
print(f'Bad subjects ({EX_TAG}): {sorted(PROBLEMATIC[EXERCISE])}')

In [ ]:
def find_files(root,ex_num):
    es_tag=f'Es{ex_num}'
    prefixes=('E_ID','NE_ID','P_ID','B_ID','S_ID')
    results=[]
    for raw_dir in glob.glob(os.path.join(root,'**',es_tag,'Raw'),recursive=True):
        parts=raw_dir.replace('\\','/').split('/')
        sid=next((p for p in parts if any(p.startswith(px) for px in prefixes)),None)
        if sid is None: continue
        csv_files=glob.glob(os.path.join(raw_dir,'JointOrientation*.csv'))
        if not csv_files: continue
        csv_path=max(csv_files,key=os.path.getsize)
        es_dir=os.path.dirname(raw_dir)
        xlsx_path=os.path.join(es_dir,'Label',f'ClinicalAssessment_{sid}.xlsx')
        if not os.path.exists(xlsx_path) or os.path.getsize(xlsx_path) < 15_000: continue
        results.append((sid,csv_path,xlsx_path))
    return results
all_files=find_files(ROOT,EXERCISE)
print(f'Found {len(all_files)} valid pairs (expected ~{N_SAMPLES//3} subjects)')

In [ ]:
def extract_label(xlsx_path,label_col):
    try:
        df=pd.read_excel(xlsx_path,sheet_name='Foglio1',engine='openpyxl')
        if label_col in df.columns:
            vals=df[label_col].dropna().values
            if len(vals) > 0:
                v=float(vals[0])
                if 0 <= v <= 50: 
                    return v
        for col in df.columns:
            try:
                vals=pd.to_numeric(df[col],errors='coerce').dropna().values
                if len(vals) > 0 and 0 <= float(vals[0]) <= 50: 
                    return float(vals[0])
            except Exception: pass
    except Exception: pass
    return None


In [ ]:
def load_and_reorder(csv_path):
    df=pd.read_csv(csv_path,header=None)
    data=df.iloc[:,:100].values.astype(np.float32)
    if len(data) > 30: data=data[15:-15]
    valid=[c for c in COL_ORDER_CLEAN if c < data.shape[1]]
    data=data[:,valid]
    if data.shape[1] < 80:
        pad=np.zeros((data.shape[0],80-data.shape[1]),dtype=np.float32)
        data=np.concatenate([data,pad],axis=1)
    return data[:,:80]
def auto_segment(data,ex_num,n_reps=3):
    T=data.shape[0]
    ch=min(EXERCISE_CHANNELS[ex_num],data.shape[1]-1)
    sig=gaussian_filter1d(data[:,ch].astype(float),sigma=5)
    sn=sig-sig.mean()
    if sn.std() > 1e-6:
        ac=np.correlate(sn,sn,mode='full')[len(sn)-1:]
        ac/=(ac[0]+1e-8)
        peaks,_=find_peaks(ac,height=0.2,distance=20)
        period=int(peaks[0]) if len(peaks) > 0 else T//3
    else:
        period=T//3
    n_det=max(3,min(8,T//max(period,1)))
    split=T//n_det
    bounds=[i*split for i in range(n_det+1)]; bounds[-1]=T
    vel=np.abs(np.diff(sig,prepend=sig[0]))
    vel_sm=gaussian_filter1d(vel,sigma=3)
    def snap(idx,w=15):
        lo,hi=max(0,idx-w),min(T,idx+w)
        return lo+int(np.argmin(vel_sm[lo:hi]))
    snapped=sorted(set([0]+[snap(b) for b in bounds[1:-1]]+[T]))
    segs=[data[snapped[i]:snapped[i+1]] for i in range(len(snapped)-1) if snapped[i+1]-snapped[i]>=20]
    if len(segs) < n_reps:
        s=T//n_reps; segs=[data[i*s:(i+1)*s] for i in range(n_reps)]
    return segs[:n_reps]
def resample_to_length(seg,target=100):
    T=len(seg)
    if T==target: return seg
    xo,xn=np.linspace(0,1,T),np.linspace(0,1,target)
    out=np.zeros((target,seg.shape[1]),dtype=np.float32)
    for c in range(seg.shape[1]):
        out[:,c]=interp1d(xo,seg[:,c],kind='linear')(xn)
    return out
print('Preprocessing helpers ready.')

In [ ]:
if os.path.exists(NPY_X) and os.path.exists(NPY_Y):
    print('Loading cached arrays...')
    X_raw=np.load(NPY_X); y_raw=np.load(NPY_Y)
else:
    print(f'Building {EX_TAG} dataset...')
    bad=PROBLEMATIC[EXERCISE]
    X_list,y_list,skipped=[],[],[]
    for sid,csv_path,xlsx_path in all_files:
        if sid in bad: 
            continue
        score=extract_label(xlsx_path,LABEL_COL)
        if score is None: skipped.append((sid,'no label')); continue
        try: data=load_and_reorder(csv_path)
        except Exception as e: skipped.append((sid,str(e))); continue
        if data.shape[0] < 60: skipped.append((sid,f'short:{data.shape[0]}')); continue
        for seg in auto_segment(data,EXERCISE,n_reps=3):
            X_list.append(resample_to_length(seg,target=100)); y_list.append(score)
    X_raw=np.array(X_list,dtype=np.float32)
    y_raw=np.array(y_list,dtype=np.float32)
    np.save(NPY_X,X_raw); np.save(NPY_Y,y_raw)
    if skipped: print(f'Skipped {len(skipped)}: {skipped[:5]}')
print(f'X={X_raw.shape}  y={y_raw.shape}')

In [ ]:
idx_all=np.arange(len(X_raw))
idx_tmp,idx_test=train_test_split(idx_all,test_size=0.20,random_state=42)
idx_train,idx_val=train_test_split(idx_tmp,test_size=0.25,random_state=42)
def smooth_sample(s):
    out=s.copy()
    for ch in range(out.shape[1]):
        out[:,ch]=gaussian_filter1d(out[:,ch].astype(float),sigma=5)
    return out.astype(np.float32)
def preprocess_split(raw,idx_tr,idx_va,idx_te):
    tr_raw=raw[idx_tr].astype(np.float32)
    tr_mean=tr_raw.mean(axis=(0,1),keepdims=True)
    tr_centered=tr_raw-tr_mean
    xmin=float(tr_centered.min())
    xmax=float(tr_centered.max())
    def norm(arr):
        c=arr.astype(np.float32)-tr_mean
        return 2.0*(c-xmin)/(xmax-xmin+1e-8)-1.0
    Xtr=np.stack([smooth_sample(norm(raw[i:i+1])[0]) for i in idx_tr])
    Xva=np.stack([smooth_sample(norm(raw[i:i+1])[0]) for i in idx_va])
    Xte=np.stack([smooth_sample(norm(raw[i:i+1])[0]) for i in idx_te])
    return Xtr,Xva,Xte
Xtr_un,Xva_un,Xte_un=preprocess_split(X_raw,idx_train,idx_val,idx_test)
def pad104(a):
    return np.concatenate([a[:,:2,:],a,a[:,-3:-1,:]],axis=1)
X_train=pad104(Xtr_un); X_val=pad104(Xva_un); X_test=pad104(Xte_un)
Xu_train=Xtr_un
Xu_val=Xva_un
Xu_test=Xte_un
assert X_train.shape[1]== 104 and X_train.shape[1] % 8== 0
y_raw_tr=y_raw[idx_train]
y_min,y_max=float(y_raw_tr.min()),float(y_raw_tr.max())
y_train=((y_raw[idx_train]-y_min)/(y_max-y_min+1e-8)).astype(np.float32)
y_val=((y_raw[idx_val]-y_min)/(y_max-y_min+1e-8)).astype(np.float32)
y_test=((y_raw[idx_test]-y_min)/(y_max-y_min+1e-8)).astype(np.float32)
print(f'Train={X_train.shape[0]}  Val={X_val.shape[0]}  Test={X_test.shape[0]}')
print(f'y_train range: [{y_train.min():.3f},{y_train.max():.3f}]')

In [ ]:
def ms(a): 
    return a,a[:,::2,:],a[:,::4,:],a[:,::8,:]
Xtr1,Xtr2,Xtr4,Xtr8=ms(X_train)
Xva1,Xva2,Xva4,Xva8=ms(X_val)
Xte1,Xte2,Xte4,Xte8=ms(X_test)
print(f'Scales: {Xtr1.shape} {Xtr2.shape} {Xtr4.shape} {Xtr8.shape}')
SK_TR=[Xtr1,Xtr2,Xtr4,Xtr8]; SK_VA=[Xva1,Xva2,Xva4,Xva8]; SK_TE=[Xte1,Xte2,Xte4,Xte8]

In [ ]:
PF_DIM=3
def compute_rep_features(X_arr):
    diffs=np.abs(np.diff(X_arr,axis=1))                   
    smoothness=diffs.mean(axis=(1,2))                            
    rom=(X_arr.max(axis=1)-X_arr.min(axis=1)).mean(axis=1) 
    rep_var=X_arr.var(axis=(1,2))                         
    return np.stack([smoothness,rom,rep_var],axis=1).astype(np.float32)
Pf_raw_tr=compute_rep_features(Xu_train)
Pf_raw_va=compute_rep_features(Xu_val)
Pf_raw_te=compute_rep_features(Xu_test)
pf_mean=Pf_raw_tr.mean(axis=0,keepdims=True)
pf_std=Pf_raw_tr.std(axis=0,keepdims=True)+1e-8
Pf_tr=(Pf_raw_tr-pf_mean)/pf_std
Pf_va=(Pf_raw_va-pf_mean)/pf_std
Pf_te=(Pf_raw_te-pf_mean)/pf_std
print(f'Rep features (train): {Pf_tr.shape}  '
      f'mean={Pf_tr.mean(0).round(3)}  std={Pf_tr.std(0).round(3)}')
print(f'Rep features (val):   {Pf_va.shape}')
print(f'Rep features (test):  {Pf_te.shape}')
FULL_TR=SK_TR+[Pf_tr]
FULL_VA=SK_VA+[Pf_va]
FULL_TE=SK_TE+[Pf_te]
def build_subject_groups(idx_split,n_reps=3):
    sorted_idx=np.sort(idx_split)
    groups,i=[],0
    while i+n_reps-1 < len(sorted_idx):
        triplet=sorted_idx[i:i+n_reps]
        if triplet[-1]-triplet[0]== n_reps-1:
            positions=[int(np.where(idx_split== t)[0][0]) for t in triplet]
            groups.append(positions)
            i+= n_reps
        else:
            i+= 1
    return groups
TRAIN_GROUPS=build_subject_groups(idx_train,n_reps=3)
print(f'Subject triplet groups in train: {len(TRAIN_GROUPS)} '
      f'({len(TRAIN_GROUPS)*3}/{len(idx_train)} samples covered)')
print('N3 rep-level features ready — no test-set leakage.')

In [ ]:
def make_tcn(nb_filters,dilations,padding='causal',return_sequences=True,dropout=0.05):
    return TCN(nb_filters=nb_filters,kernel_size=3,dilations=dilations,
               padding=padding,dropout_rate=dropout,
               return_sequences=return_sequences,
               use_batch_norm=False,use_layer_norm=True)
def body_part_subnetwork(x1,x2,x4,x8,lo,hi,dropout,tag,nb_f=12):
    sl=lambda inp,sfx: layers.Lambda(lambda t: t[:,:,lo:hi],name=f'sl_{tag}_{sfx}')(inp)
    s1,s2,s4,s8=sl(x1,'s1'),sl(x2,'s2'),sl(x4,'s4'),sl(x8,'s8')
    t1=make_tcn(nb_f,[1,2,4,8,16,32],padding='causal',dropout=dropout)(s1)
    t2=make_tcn(nb_f,[1,2,4,8,16,32],padding='causal',dropout=dropout)(s2)
    t4=make_tcn(nb_f,[1,2,4,8,16,32],padding='causal',dropout=dropout)(s4)
    t8=make_tcn(nb_f,[1,2,4,8,16,32],padding='causal',dropout=dropout)(s8)
    return layers.Concatenate(axis=1)([t1,t2,t4,t8])
def temporal_attention_pool(x,named=False):
    logits=layers.Dense(1)(x)
    kw=dict(name='attn_weights') if named else {}
    weights=layers.Softmax(axis=1,**kw)(logits)
    pooled=layers.Lambda(lambda xw: tf.reduce_sum(xw[0] * xw[1],axis=1))([x,weights])
    return pooled,weights
def part_se_gate(parts,nb_f=12,named=False):
    pooled=[layers.GlobalAveragePooling1D()(p) for p in parts]
    concat=layers.Concatenate(axis=-1)(pooled)
    kw=dict(name='part_importance') if named else {}
    gates=layers.Dense(5,activation='sigmoid',**kw)(concat)
    gated=[]
    for i,p in enumerate(parts):
        gi=layers.Lambda(lambda gv,idx=i: tf.expand_dims(tf.expand_dims(gv[:,idx],1),1))(gates)
        gated.append(layers.Multiply()([p,gi]))
    return gated
print('Architecture helpers ready.')

In [ ]:
def build_baseline():
    nb_f=12
    i1=layers.Input((104,80),name='x1'); i2=layers.Input((52,80),name='x2')
    i4=layers.Input((26,80),name='x4'); i8=layers.Input((13,80),name='x8')
    parts=[body_part_subnetwork(i1,i2,i4,i8,lo,hi,DROPOUT,f'bl{lo}',nb_f=nb_f) for lo,hi in [(0,16),(16,32),(32,48),(48,64),(64,80)]]
    x=layers.Concatenate(axis=-1)(parts)
    x=make_tcn(nb_f,[1,2,4,8],padding='causal',dropout=DROPOUT)(x)
    x=make_tcn(12,[1,2,4],padding='causal',dropout=DROPOUT)(x)
    x=make_tcn(8,[1,2],padding='causal',return_sequences=True,dropout=DROPOUT)(x)
    x=layers.GlobalAveragePooling1D()(x)
    score=layers.Dense(1,activation='linear')(x)
    return Model(inputs=[i1,i2,i4,i8],outputs=score,name='LightPRA_Baseline')
def build_n1():
    nb_f=12
    i1=layers.Input((104,80),name='x1'); i2=layers.Input((52,80),name='x2')
    i4=layers.Input((26,80),name='x4'); i8=layers.Input((13,80),name='x8')
    parts=[body_part_subnetwork(i1,i2,i4,i8,lo,hi,DROPOUT,f'n1{lo}',nb_f=nb_f) for lo,hi in [(0,16),(16,32),(32,48),(48,64),(64,80)]]
    x=layers.Concatenate(axis=-1)(parts)
    x=make_tcn(nb_f,[1,2,4,8],padding='causal',dropout=DROPOUT)(x)
    x=make_tcn(12,[1,2,4],padding='causal',dropout=DROPOUT)(x)
    x=make_tcn(8,[1,2],padding='causal',return_sequences=True,dropout=DROPOUT)(x)
    x,_=temporal_attention_pool(x,named=True)
    score=layers.Dense(1,activation='linear',name='quality_score')(x)
    return Model(inputs=[i1,i2,i4,i8],outputs=score,name='LightPRA_N1')
def build_n2():
    nb_f=12
    i1=layers.Input((104,80),name='x1'); i2=layers.Input((52,80),name='x2')
    i4=layers.Input((26,80),name='x4'); i8=layers.Input((13,80),name='x8')
    parts=[body_part_subnetwork(i1,i2,i4,i8,lo,hi,DROPOUT,f'n2{lo}',nb_f=nb_f) for lo,hi in [(0,16),(16,32),(32,48),(48,64),(64,80)]]
    gated=part_se_gate(parts,nb_f=nb_f,named=True)
    x=layers.Concatenate(axis=-1)(gated)
    x=make_tcn(nb_f,[1,2,4,8],padding='causal',dropout=DROPOUT)(x)
    x=make_tcn(12,[1,2,4],padding='causal',dropout=DROPOUT)(x)
    x=make_tcn(8,[1,2],padding='causal',return_sequences=True,dropout=DROPOUT)(x)
    x=layers.GlobalAveragePooling1D()(x)
    score=layers.Dense(1,activation='linear',name='quality_score')(x)
    return Model(inputs=[i1,i2,i4,i8],outputs=score,name='LightPRA_N2')
def build_n3():
    nb_f=12
    i1=layers.Input((104,80),name='x1'); i2=layers.Input((52,80),name='x2')
    i4=layers.Input((26,80),name='x4'); i8=layers.Input((13,80),name='x8')
    ipf=layers.Input((PF_DIM,),name='pf')
    parts=[body_part_subnetwork(i1,i2,i4,i8,lo,hi,DROPOUT,f'n3{lo}',nb_f=nb_f) for lo,hi in [(0,16),(16,32),(32,48),(48,64),(64,80)]]
    x=layers.Concatenate(axis=-1)(parts)
    x=make_tcn(nb_f,[1,2,4,8],padding='causal',dropout=DROPOUT)(x)
    x=make_tcn(12,[1,2,4],padding='causal',dropout=DROPOUT)(x)
    x=make_tcn(8,[1,2],padding='causal',return_sequences=True,dropout=DROPOUT)(x)
    x=layers.GlobalAveragePooling1D()(x)
    pf=layers.Dense(8,activation='relu')(ipf)
    pf=layers.Dropout(0.1)(pf)
    pf=layers.Dense(4,activation='relu')(pf)
    fused=layers.Concatenate(name='n3_fusion')([x,pf])
    fused=layers.Dense(8,activation='relu')(fused)
    score=layers.Dense(1,activation='linear',name='quality_score')(fused)
    return Model(inputs=[i1,i2,i4,i8,ipf],outputs=score,name='LightPRA_N3')
def build_final():
    nb_f=12
    i1=layers.Input((104,80),name='x1'); i2=layers.Input((52,80),name='x2')
    i4=layers.Input((26,80),name='x4'); i8=layers.Input((13,80),name='x8')
    ipf=layers.Input((PF_DIM,),name='pf')
    parts=[body_part_subnetwork(i1,i2,i4,i8,lo,hi,DROPOUT,f'fn{lo}',nb_f=nb_f) for lo,hi in [(0,16),(16,32),(32,48),(48,64),(64,80)]]
    gated=part_se_gate(parts,nb_f=nb_f,named=True)    # N2
    x=layers.Concatenate(axis=-1)(gated)
    x=make_tcn(nb_f,[1,2,4,8],padding='causal',dropout=DROPOUT)(x)
    x=make_tcn(12,[1,2,4],padding='causal',dropout=DROPOUT)(x)
    x=make_tcn(8,[1,2],padding='causal',return_sequences=True,dropout=DROPOUT)(x)
    x,_=temporal_attention_pool(x,named=True)          # N1
    pf=layers.Dense(8,activation='relu')(ipf)           # N3 branch
    pf=layers.Dropout(0.1)(pf)
    pf=layers.Dense(4,activation='relu')(pf)
    fused=layers.Concatenate(name='n3_fusion')([x,pf])
    fused=layers.Dense(8,activation='relu')(fused)
    score=layers.Dense(1,activation='linear',name='quality_score')(fused)
    return Model(inputs=[i1,i2,i4,i8,ipf],outputs=score,name='LightPRA_FINAL')
bl_params=build_baseline().count_params()
n1_params=build_n1().count_params()
n2_params=build_n2().count_params()
n3_params=build_n3().count_params()
fin_params=build_final().count_params()
print(f'Baseline params: {bl_params:,}')
print(f'N1 params:       {n1_params:,}  (delta: +{n1_params-bl_params})')
print(f'N2 params:       {n2_params:,}  (delta: +{n2_params-bl_params})')
print(f'N3 params:       {n3_params:,}  (delta: +{n3_params-bl_params})')
print(f'FINAL params:    {fin_params:,}  (delta: +{fin_params-bl_params})')

In [ ]:
BASELINE_NAME='BASELINE'
N1_NAME='N1 (Temporal Attn)'
N2_NAME='N2 (Part SE Gate)'
N3_NAME='N3 (Rep-Aware Scoring)'
FINAL_NAME='FINAL LightPRA-XAT'
CONSISTENCY_LAMBDA=0.3
def _make_dataset(x_list,y,batch_size=7):
    inputs=tuple(tf.constant(a,dtype=tf.float32) for a in x_list)
    ys=tf.constant(y,dtype=tf.float32)
    ds=tf.data.Dataset.from_tensor_slices((inputs,ys))
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
@tf.function
def _consist_step(model,all_sk,groups_t,lam,training=True):
    with tf.GradientTape() as tape:
        all_preds=tf.squeeze(model(all_sk,training=training),axis=-1)
        var_losses=tf.map_fn(
            lambda triplet: tf.math.reduce_variance(tf.gather(all_preds,triplet)),
            groups_t,fn_output_signature=tf.float32)
        consist_loss=lam * tf.reduce_mean(var_losses)
    grads=tape.gradient(consist_loss,model.trainable_variables)
    model.optimizer.apply_gradients(zip(grads,model.trainable_variables))
    return consist_loss
def train_and_eval(model,x_tr,x_va,x_te,name='',epochs=400,patience=75,use_consistency=False):
    loss_fn=keras.losses.MeanAbsoluteError()
    model.compile(optimizer=keras.optimizers.Adam(1e-3),loss=loss_fn)
    ds_tr=_make_dataset(x_tr,y_train,batch_size=7)
    ds_va=_make_dataset(x_va,y_val,batch_size=7)
    best_val=np.inf; patience_count=0; best_w=None
    if use_consistency and len(TRAIN_GROUPS) > 0:
        sk_arrays=[tf.constant(x_tr[k],dtype=tf.float32) for k in range(4)]
        if len(model.inputs)==5:
            dummy_pf=tf.zeros((len(idx_train),PF_DIM),dtype=tf.float32)
            all_sk=tuple(sk_arrays)+(dummy_pf,)
        else:
            all_sk=tuple(sk_arrays)
        groups_t=tf.constant(TRAIN_GROUPS,dtype=tf.int32)
        lam=tf.constant(CONSISTENCY_LAMBDA,dtype=tf.float32)
    for ep in range(epochs):
        model.fit(ds_tr,verbose=0,epochs=1)
        if use_consistency and len(TRAIN_GROUPS) > 0:
            _consist_step(model,all_sk,groups_t,lam)
        val_loss=float(model.evaluate(ds_va,verbose=0))
        if val_loss < best_val-1e-6:
            best_val=val_loss; patience_count=0
            best_w=[v.numpy().copy() for v in model.trainable_variables]
        else:
            patience_count += 1
            if patience_count >= patience:
                break
    if best_w is not None:
        for var,w in zip(model.trainable_variables,best_w):
            var.assign(w)
    y_pred=model.predict(x_te,verbose=0).reshape(-1)
    mad=float(np.mean(np.abs(y_pred-y_test)))
    rmse=float(np.sqrt(np.mean((y_pred-y_test)**2)))
    if name:
        print(f'  {name:<35} MAD={mad:.4f}  RMSE={rmse:.4f}')
    return mad,rmse,y_pred
variants={}
print(f'Training all variants for {EX_TAG} ...')
t_all=time.time()
keras.backend.clear_session(); gc.collect(); _set_seed(42)
t0=time.time()
m,r,p=train_and_eval(build_baseline(),SK_TR,SK_VA,SK_TE,BASELINE_NAME)
variants[BASELINE_NAME]=dict(mad=m,rmse=r,pred=p)
print(f'    ({time.time()-t0:.0f}s)')
keras.backend.clear_session(); gc.collect(); _set_seed(42)
t0=time.time()
m,r,p=train_and_eval(build_n1(),SK_TR,SK_VA,SK_TE,N1_NAME)
variants[N1_NAME]=dict(mad=m,rmse=r,pred=p)
print(f'    ({time.time()-t0:.0f}s)')
keras.backend.clear_session(); gc.collect(); _set_seed(42)
t0=time.time()
m,r,p=train_and_eval(build_n2(),SK_TR,SK_VA,SK_TE,N2_NAME)
variants[N2_NAME]=dict(mad=m,rmse=r,pred=p)
print(f'    ({time.time()-t0:.0f}s)')
keras.backend.clear_session(); gc.collect(); _set_seed(42)
t0=time.time()
m,r,p=train_and_eval(build_n3(),FULL_TR,FULL_VA,FULL_TE,N3_NAME)
variants[N3_NAME]=dict(mad=m,rmse=r,pred=p)
print(f'    ({time.time()-t0:.0f}s)')
keras.backend.clear_session(); gc.collect(); _set_seed(42)
t0=time.time()
model=build_final()
m,r,p=train_and_eval(model,FULL_TR,FULL_VA,FULL_TE,FINAL_NAME,use_consistency=True)
variants[FINAL_NAME]=dict(mad=m,rmse=r,pred=p)
print(f'    ({time.time()-t0:.0f}s)')
y_pred=variants[FINAL_NAME]['pred']
mad=variants[FINAL_NAME]['mad']
rmse=variants[FINAL_NAME]['rmse']
print(f'\nAll trained in {time.time()-t_all:.0f}s')
print(f'{FINAL_NAME}: MAD={mad:.4f}  RMSE={rmse:.4f}')

In [ ]:
bl_mad=variants[BASELINE_NAME]['mad']
best_v=min(variants,key=lambda k: variants[k]['mad'])
print(f'\n=== Ablation Study-{EX_TAG}===')
print(f"{'Variant':<28} {'MAD':>8} {'RMSE':>8}  {'Delta MAD vs BL':>16}")
print('-'*62)
for name,v in variants.items():
    delta=(v['mad']-bl_mad)/(bl_mad+1e-9) * 100
    marker=' <- best' if name== best_v else ''
    print(f"  {name:<26} {v['mad']:>8.4f} {v['rmse']:>8.4f}  {delta:>+14.1f}%{marker}")

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
names=list(variants.keys())
mads=[variants[n]['mad']  for n in names]
rmses=[variants[n]['rmse'] for n in names]
colors=['#d62728' if n== BASELINE_NAME else '#2ca02c' if n== FINAL_NAME else '#1f77b4' for n in names]
for ax,vals,metric in zip(axes,[mads,rmses],['MAD down','RMSE down']):
    bars=ax.bar(range(len(names)),vals,color=colors,edgecolor='white',linewidth=1.2)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names,rotation=25,ha='right',fontsize=9)
    ax.set_ylabel(metric)
    ax.set_title(f'Ablation-{metric} [{EX_TAG}]')
    for bar,v in zip(bars,vals):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.001,
                f'{v:.4f}',ha='center',va='bottom',fontsize=8)
    ax.set_ylim(0,max(vals) * 1.30)
axes[0].legend(handles=[Patch(color='#d62728',label='BASELINE'),Patch(color='#1f77b4',label='N1/N2/N3'),Patch(color='#2ca02c',label='FINAL LightPRA-XAT')],fontsize=9)
plt.suptitle(f'Ablation Study [{EX_TAG}]',fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'ablation_bar_{EX_TAG}.png'),dpi=150)
plt.show()

In [ ]:
nv=len(variants); ncols=3; nrows=(nv+ncols-1)//ncols
fig,axes=plt.subplots(nrows,ncols,figsize=(15,4 * nrows))
for ax,(name,v) in zip(axes.flat,variants.items()):
    clr='#d62728' if name== BASELINE_NAME else '#2ca02c' if name== FINAL_NAME else '#1f77b4'
    ax.scatter(y_test,v['pred'],alpha=0.55,s=18,color=clr)
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_title(f"{name}\nMAD={v['mad']:.4f}  RMSE={v['rmse']:.4f}",fontsize=9)
    ax.set_xlabel('True')
    ax.set_ylabel('Predicted')
for ax in axes.flat[nv:]:
    ax.set_visible(False)
plt.suptitle(f'Ablation Scatter [{EX_TAG}]',fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'ablation_scatter_{EX_TAG}.png'),dpi=150)
plt.show()


In [ ]:
attn_m=Model(model.inputs,model.get_layer('attn_weights').output)
order=np.argsort(y_test)
show_idx=list(order[:2])+list(order[-2:])
fig,axes=plt.subplots(2,2,figsize=(13,6))
for ax,idx in zip(axes.flat,show_idx):
    xi=[a[idx:idx+1] for a in FULL_TE]
    aw=attn_m.predict(xi,verbose=0)
    sc=float(model.predict(xi,verbose=0).reshape(-1)[0])
    ax.bar(range(aw.shape[1]),aw[0,:,0],width=1,color='steelblue',alpha=0.7)
    for xv,col,lb in [(104,'r','end x1'),(156,'orange','end x1/2'),(182,'g','end x1/4')]:
        ax.axvline(xv,color=col,lw=1,ls='--',label=lb)
    ax.set_title(f'idx {idx}  True={y_test[idx]:.3f}  Pred={sc:.3f}')
    ax.set_xlabel('Timestep'); ax.set_ylabel('Attention weight')
axes.flat[0].legend(fontsize=7)
fig.suptitle(f'N1 Temporal Attention [{EX_TAG}]',fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'attn_{EX_TAG}.png'),dpi=120)
plt.show()

In [ ]:
gate_m=Model(model.inputs,model.get_layer('part_importance').output)
all_gates=gate_m.predict(FULL_TE,verbose=0)
part_names=['Trunk','L.Arm','R.Arm','L.Leg','R.Leg']
mean_gates=all_gates.mean(0)
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].bar(part_names,mean_gates,color='coral')
axes[0].set_ylim(0,1)
axes[0].set(ylabel='Mean SE gate',title=f'Part Importance Gates-{EX_TAG}')
im=axes[1].imshow(all_gates.T,aspect='auto',cmap='YlOrRd',vmin=0,vmax=1)
axes[1].set_yticks(range(5)); axes[1].set_yticklabels(part_names)
axes[1].set(xlabel='Test sample',title='Gates per sample')
plt.colorbar(im,ax=axes[1])
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'gates_{EX_TAG}.png'),dpi=120)
plt.show()
print('Most important body part:',part_names[int(np.argmax(mean_gates))])

In [ ]:
feature_names=['Smoothness','ROM (Range of Motion)','Rep Variance']
fig,axes=plt.subplots(2,3,figsize=(14,8))
for col,fname in enumerate(feature_names):
    ax=axes[0,col]
    ax.hist(Pf_tr[:,col],bins=20,alpha=0.6,label='Train',color='steelblue')
    ax.hist(Pf_te[:,col],bins=20,alpha=0.6,label='Test',color='coral')
    ax.set_title(f'N3 Feature: {fname}')
    ax.set_xlabel('Normalised value'); ax.legend(fontsize=8)
    ax=axes[1,col]
    ax.scatter(Pf_te[:,col],y_test,alpha=0.5,s=15,color='teal')
    ax.set_xlabel(fname); ax.set_ylabel('True Score')
    corr,_=pearsonr(Pf_te[:,col],y_test)
    ax.set_title(f'Corr with score: {corr:.3f}')
plt.suptitle(f'N3 Rep-Level Features [{EX_TAG}]',fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'n3_features_{EX_TAG}.png'),dpi=120)
plt.show()
def get_rep_variance(model_preds,idx_split,n_reps=3):
    sorted_idx=np.sort(idx_split)
    variances,i=[],0
    while i+n_reps-1 < len(sorted_idx):
        triplet=sorted_idx[i:i+n_reps]
        if triplet[-1]-triplet[0]== n_reps-1:
            pos=[int(np.where(idx_split== t)[0][0]) for t in triplet]
            variances.append(float(np.var(model_preds[pos])))
            i+=n_reps
        else:
            i+=1
    return np.array(variances)
bl_var=get_rep_variance(variants[BASELINE_NAME]['pred'],idx_test)
n3_var=get_rep_variance(variants[N3_NAME]['pred'],idx_test)
fin_var=get_rep_variance(variants[FINAL_NAME]['pred'],idx_test)
means_v=[a.mean() if len(a) else 0.0 for a in [bl_var,n3_var,fin_var]]
labels_v=[BASELINE_NAME,N3_NAME,FINAL_NAME]
fig,axes=plt.subplots(1,2,figsize=(13,4))
axes[0].bar(labels_v,means_v,color=['#d62728','#ff7f0e','#2ca02c'])
axes[0].set_ylabel('Mean prediction variance across 3 reps')
axes[0].set_title(f'N3 Effect: Within-Subject Rep Consistency [{EX_TAG}]\n'
                   f'(lower=model more consistent across reps)')
for i,v in enumerate(means_v):
    axes[0].text(i,v+1e-5,f'{v:.4f}',ha='center',fontsize=9)
axes[1].scatter(y_test,variants[N3_NAME]['pred'],alpha=0.6,s=20,color='#ff7f0e',label=N3_NAME)
axes[1].scatter(y_test,variants[BASELINE_NAME]['pred'],alpha=0.4,s=20,color='#d62728',label=BASELINE_NAME)
axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set_xlabel('True Score'); axes[1].set_ylabel('Predicted Score')
axes[1].set_title(f'N3 vs Baseline [{EX_TAG}]')
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'n3_consistency_{EX_TAG}.png'),dpi=150)
plt.show()
print(f'Mean rep variance  Baseline:{means_v[0]:.4f}  N3:{means_v[1]:.4f}  FINAL:{means_v[2]:.4f}')

In [ ]:
print(f'\n=== N3 Rep-Feature Correlation with |Error|===')
print(f"{'Feature':<26} {'Corr':>10}  {'p-value':>10}  note")
print('-'*60)
n3_errs=np.abs(variants[N3_NAME]['pred']-y_test)
for i,fname in enumerate(['Smoothness','ROM','Rep Variance']):
    r,p=pearsonr(Pf_te[:,i],n3_errs)
    note='hurts accuracy' if r > 0 else 'helps accuracy'
    print(f'  {fname:<24} {r:>+10.4f}  {p:>10.4f}  ({note})')
bl_errs=np.abs(variants[BASELINE_NAME]['pred']-y_test)
improved=(n3_errs < bl_errs).mean() * 100
print(f'\nN3 improves on {improved:.1f}% of test samples vs Baseline')

In [ ]:
def extended_metrics(y_true,yp):
    mad=float(np.mean(np.abs(yp-y_true)))
    rmse=float(np.sqrt(np.mean((yp-y_true) ** 2)))
    r,_=pearsonr(y_true,yp)
    mape=float(np.mean(np.abs((yp-y_true)/(y_true+1e-8))) * 100)
    return dict(MAD=mad,RMSE=rmse,PCC=r,MAPE=mape)
bl_m=extended_metrics(y_test,variants[BASELINE_NAME]['pred'])
fin_m=extended_metrics(y_test,variants[FINAL_NAME]['pred'])
print(f"\n{'-'*60}\n  OVERALL COMPARISON-{EX_TAG}\n{'-'*60}")
print(f"{'Metric':<10} {'Baseline':>18} {'FINAL LightPRA-XAT':>20} {'Delta':>10}")
print('-'*60)
for k in ['MAD','RMSE','MAPE','PCC']:
    bv,xv=bl_m[k],fin_m[k]
    note='(lower is better)' if k != 'PCC' else '(higher is better)'
    print(f"  {k:<8} {bv:>18.4f} {xv:>20.4f} {xv-bv:>+10.4f}  {note}")
print('-'*60)
fig,axes=plt.subplots(1,2,figsize=(12,5))
for ax,name,clr in [(axes[0],BASELINE_NAME,'#d62728'),(axes[1],FINAL_NAME,'#2ca02c')]:
    preds=variants[name]['pred']
    m=extended_metrics(y_test,preds)
    ax.scatter(y_test,preds,alpha=0.6,s=20,color=clr)
    ax.plot([0,1],[0,1],'k--',lw=1)
    ax.set_title(f"{name}\nMAD={m['MAD']:.4f}  RMSE={m['RMSE']:.4f}  PCC={m['PCC']:.3f}",fontsize=10)
    ax.set_xlabel('True Score')
    ax.set_ylabel('Predicted Score')
plt.suptitle(f'Overall Comparison-{EX_TAG}',fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR,f'final_comparison_{EX_TAG}.png'),dpi=150)
plt.show()
print('\nSaved outputs:')
for fn in sorted(os.listdir(WORK_DIR)):
    if EX_TAG in fn:
        print(f'  {fn:<60} {os.path.getsize(os.path.join(WORK_DIR,fn))/1024:>7.1f} KB')